# Day 5 — Transit Readiness Briefing

**Estimated time:** 2:00 PM – 5:00 PM  •  **Prereq:** Day 4 gap grid

Today you write the briefing. This notebook is **mostly markdown**
— the numbers come from the checkpoint files you've already produced.

The briefing is a one-page answer aimed at a decision-maker (someone
on MARTA's planning team, or a city official thinking about World
Cup readiness). Evidence-backed. Honest about what we didn't model.
Short.

## Structure

1. The question
2. The data we used
3. What we found — the 3×3 grid
4. Cross-checks — Super Bowl LIII and the GT regression
5. Our answer
6. What we didn't model
7. What we'd study next


## 1. Setup — load every checkpoint


In [ ]:
# Setup — run this first. In Colab: clones the repo and installs deps.
# Locally: no-op (the conda env handles it).
import os, sys, urllib.request
try:
    import google.colab  # noqa: F401
    if not os.path.exists('/tmp/colab_bootstrap.py'):
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/arctic-gsu/arctic-scisynth-2026-rwd-public/main/setup/colab_bootstrap.py',
            '/tmp/colab_bootstrap.py',
        )
    sys.path.insert(0, '/tmp')
    from colab_bootstrap import bootstrap
    bootstrap()
except ImportError:
    pass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'raw').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'raw').is_dir(), (
    f'❌ Could not find data/raw from {Path.cwd()}. '
    'If you are in Colab, re-run this cell — the bootstrap may not have completed.'
)
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
CHECKPOINTS = ROOT / 'data' / 'checkpoints'

MARTA_NTD_ID = 40022
MODE_NAMES = {'HR': 'Heavy Rail', 'MB': 'Bus', 'SR': 'Streetcar',
              'DR': 'Demand Response'}
STADIUM_ADJACENT = ['sec district', 'vine city']
STADIUM_AREA = STADIUM_ADJACENT + ['five points']

# World Cup demand model (Day 4). Canonical values; sources cited in
# the Day 4 notebook narrative. Changing any value here propagates
# through Day 4 and Day 5 with no duplicate literals to update.
STADIUM_CAPACITY = 75_000
MATCH_DAY_TRAINS_PER_HOUR = 24
ARRIVAL_WINDOW = 2.5
MODE_SHARE = {'low': 0.15, 'central': 0.22, 'high': 0.35}
TRAIN_CAP = {'seated_only': 232, 'service_standard': 400, 'full_crush': 750}
MS_ORDER = ('low', 'central', 'high')
CAP_ORDER = ('seated_only', 'service_standard', 'full_crush')
# GT regression (Santanam et al. 2021, arXiv:2106.05359; MAPE 11.7%).
GT_INTERCEPT = -1201
GT_SLOPE = 0.1739

print(f'✅ Imports ready. Working from {ROOT}')


In [ ]:
# Everything the briefing needs, all in one place.
stops         = pd.read_csv(CHECKPOINTS / 'day1_output_stops.csv')
ridership     = pd.read_csv(
    CHECKPOINTS / 'day2_output_cleaned_ridership.csv', parse_dates=['date']
)
station_freq  = pd.read_csv(CHECKPOINTS / 'day2_output_station_freq_clean.csv')
eda_summary   = pd.read_csv(CHECKPOINTS / 'day3_output_eda_summary.csv')
gap_df        = pd.read_csv(CHECKPOINTS / 'day4_output_demand_estimates.csv')
wc_ref        = pd.read_csv(PROCESSED / 'worldcup_reference.csv')

print(f'stops:        {len(stops)} rows')
print(f'ridership:    {len(ridership):,} rows')
print(f'station_freq: {len(station_freq)} rows')
print(f'eda_summary:  {len(eda_summary)} rows')
print(f'gap_df:       {len(gap_df)} scenarios')


## 2. Key numbers — the briefing's raw material


In [ ]:
print('=' * 60)
print('TRANSIT READINESS BRIEFING — KEY NUMBERS')
print('=' * 60)

# System
n_lines = station_freq['line'].nunique()
lines = sorted(station_freq['line'].dropna().unique())
print(f'\n1. Network: {len(station_freq)} rail stations, {n_lines} lines '
      f'({", ".join(lines)})')

# Stadium access
adjacent = station_freq[station_freq['station_name'].str.lower().str.contains(
    '|'.join(STADIUM_ADJACENT)
)]
print(f'\n2. Stadium-adjacent stations ({len(adjacent)}):')
for _, s in adjacent.iterrows():
    print(f"     {s['station_name']}: {s['peak_trains_per_hour']} peak / "
          f"{s['offpeak_trains_per_hour']} offpeak trains/hr (both dir)")

# Ridership
recent = ridership[(ridership['mode'] == 'HR') & (ridership['date'] >= '2024-01-01')]
avg_monthly = recent['upt'].mean()
avg_daily = avg_monthly / 22
print(f'\n3. Recent rail ridership: {avg_monthly:,.0f} monthly / '
      f'{avg_daily:,.0f} daily (weekday avg)')

# Gap grid
pivot = gap_df.pivot_table(
    index='mode_share_scenario', columns='capacity_scenario',
    values='gap_riders', aggfunc='first',
)[list(CAP_ORDER)]
print(f'\n4. Gap grid (riders):')
print(pivot.to_string())
n_s = (gap_df['verdict'] == 'SURPLUS').sum()
n_d = (gap_df['verdict'] == 'DEFICIT').sum()
print(f'   → {n_s}/9 surplus, {n_d}/9 deficit')

# Tipping point
worst = gap_df.loc[gap_df['gap_riders'].idxmin()]
print(f'\n5. Tipping point: {worst["mode_share_scenario"]} × '
      f'{worst["capacity_scenario"]} = {worst["gap_riders"]:+,d} riders')

# Anchors
gt_baseline = -1201 + 0.1739 * 70_000
central_demand = 75_000 * 0.22
uplift_pct = (central_demand / gt_baseline - 1) * 100
print(f'\n6. Cross-checks:')
print(f'   Super Bowl LIII (Feb 3 2019): 155,000 rail riders, 90-min egress')
print(f'   GT regression (Santanam et al. 2021): ~{gt_baseline:,.0f} at 70k')
print(f'   Our central scenario (22% × 75k): {central_demand:,.0f} riders '
      f'(~{uplift_pct:+.0f}% vs GT)')


## 3. The briefing template

Write your briefing in the markdown cells below. Each section has a
prompt; delete the prompt and replace it with your own prose, pulling
specific numbers from the cell above. Target: **400–600 words total**.


### 3.1 The question

> *Prompt:* In one paragraph, state the question. When are the matches?
> Where? How many fans per match? What's at stake if MARTA can't handle
> the load? Why is this a 2026 question and not a hypothetical?
>
> *Replace this prompt with your answer.*


### 3.2 The data

> *Prompt:* In a short bulleted list, name every dataset you used.
> For each, name the source (MARTA, NTD, Census, ACS), the time window,
> and the one most important column.
>
> *Replace this prompt with your answer.*


### 3.3 What we found — the 3×3 grid

> *Prompt:* Describe the grid in plain English. How many surplus?
> How many deficit? Which cell is the tipping point — what is the gap
> in riders? Reference the `pivot` table printed above. **Include a
> specific number for the tipping-point gap.**
>
> *Replace this prompt with your answer.*


### 3.4 Cross-checks

> *Prompt:* Name the two anchors — Super Bowl LIII and the Georgia
> Tech regression. How does our central scenario (16,500 riders)
> compare to what the GT regression predicts for a 70k-seat event?
> What does Super Bowl LIII tell us about operational capability
> *beyond* what the capacity grid shows?
>
> *Replace this prompt with your answer.*


### 3.5 Our answer

> *Prompt:* Given the grid and the cross-checks, what's your answer?
> "Yes" / "no" / "it depends" — whichever you pick, defend it. Name
> the specific assumption combinations under which your answer would
> flip. A strong answer names the condition under which MARTA
> succeeds and the condition under which it fails.
>
> *Replace this prompt with your answer.*


### 3.6 What we didn't model

> *Prompt:* Name at least three things this analysis does NOT cover.
> (Hint: post-game egress, fleet mix, sustained tournament demand.)
> For each, say in one sentence why it matters and why it's out of
> scope for our model.
>
> *Replace this prompt with your answer.*


### 3.7 What we'd study next

> *Prompt:* If you had another week, what's the single next study
> you'd run? Name the question, the data you'd need, and what answer
> would change your current recommendation.
>
> *Replace this prompt with your answer.*


## ✅ Final checkpoint — share your briefing

Before you leave:

1. Save the notebook (File → Save).
2. Show your instructor. They'll give you quick feedback — check
   numbers, tighten wording, flag anything mis-sourced.
3. Sit with the finished analysis for a minute. You started Monday
   with a question about a stadium and a train system; you're
   leaving Friday with an evidence-backed answer that cites the
   Census, the NTD, a Georgia Tech regression, and a published
   service-standards document.

That's real-world data analysis. Congratulations.
